[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/4-context-engineering.ipynb)

## Colab setup

Run the next cell first on Google Colab. It sparse-checkouts this public repo and installs packages only when the kernel is Colab. Local Jupyter and VS Code skip that work.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) or in an environment cell: `os.environ["OPENAI_API_KEY"] = "sk-..."`. Do that before the lesson setup cell. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

In [ ]:
# Colab setup. Run this cell before the other code cells.
# Local Jupyter and VS Code skip the clone and the install.
#
# An API key is required. On Google Colab, set OPENAI_API_KEY in one of these ways:
#   * Secrets (the key icon): a secret named OPENAI_API_KEY
#   * an environment cell: os.environ["OPENAI_API_KEY"] = "sk-..."
# Locally, put the key in the repository-root .env.
# Do not commit a key. This cell does not print the value.
# Edits you make in Colab stay in the session. They do not push to GitHub.

import os
import sys
from pathlib import Path


def in_colab():
    """True on Google Colab. False in local Jupyter and VS Code."""
    try:
        import google.colab
    except ImportError:
        return False
    return Path("/content").is_dir() and google.colab is not None


if not in_colab():
    print("Not Colab. Skipped clone and pip install.")
else:
    import shutil
    import subprocess

    missing = []
    for module_name, requirement in (
        ("openai", "openai>=1.40.0"),
        ("dotenv", "python-dotenv>=1.0.1"),
        ("httpx", "httpx>=0.27.0"),
    ):
        try:
            __import__(module_name)
        except ImportError:
            missing.append(requirement)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

    repo = Path("/content/book-agents")
    marker = repo / "tutorial" / "common" / "client.py"
    if not marker.is_file():
        if repo.exists() and not (repo / ".git").exists():
            raise RuntimeError(
                str(repo) + " exists but is not a git checkout. "
                "Move that folder aside and run this cell again."
            )
        if repo.exists():
            shutil.rmtree(repo)
        url = "https://github.com/junlinghu/book-agents.git"
        try:
            subprocess.check_call([
                "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                url, str(repo),
            ])
            subprocess.check_call([
                "git", "-C", str(repo), "sparse-checkout", "set", "tutorial",
            ])
        except subprocess.CalledProcessError:
            if repo.exists():
                shutil.rmtree(repo)
            subprocess.check_call(["git", "clone", "--depth", "1", url, str(repo)])
        if not marker.is_file():
            raise RuntimeError(
                "Colab setup could not find tutorial/common/client.py after cloning."
            )

    os.chdir(repo)
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))

    if not os.environ.get("OPENAI_API_KEY", "").strip():
        try:
            from google.colab import userdata
            secret = userdata.get("OPENAI_API_KEY")
        except Exception:
            secret = ""
        if secret and str(secret).strip():
            os.environ["OPENAI_API_KEY"] = str(secret).strip()

    if not os.environ.get("OPENAI_API_KEY", "").strip():
        raise RuntimeError(
            "OPENAI_API_KEY is empty. On Colab, add a secret named OPENAI_API_KEY "
            "(the key icon) and rerun this cell. Locally, copy .env.example to .env "
            "and paste a key from https://platform.openai.com/api-keys. Never commit .env."
        )
    print("Colab: tutorial/ is ready. OPENAI_API_KEY is set (value hidden).")


# 4. Context engineering

The loop from tutorial 3 will read whatever you stuff into the prompt. Stuffing is how a Tuesday huddle turns into context rot.

This notebook adds a note catalog. Titles go in the system prompt. Bodies come back through `read_note`. One note is over the cap so you can see the refusal. The picnic note is long and useless for a carton count. The run should not load it.

The shelf tool is imported, so the oat-milk count is checked against the database as well as the note.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, the setup cell above reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `4-context-engineering.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '4-context-engineering'
LESSON_NUMBER = 4
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.shelf import reset_db


## Tutorial 4 — notes under a budget

The prompt gets titles. `read_note` returns one short body. The picnic note is over the cap on purpose.

In [ ]:
from tutorial.common.harness import DATA, register

NOTES = DATA / "notes"
PER_NOTE_CAP = 500
TURN_BUDGET = 600


def _note_path(name):
    if not isinstance(name, str) or not name.strip():
        return "ERROR: name is required. Example: oat-milk.md"
    cleaned = name.strip().replace("\\", "/")
    if "/" in cleaned or cleaned.startswith("."):
        return "ERROR: name must be a file in the notes catalog, not a path."
    if not cleaned.endswith(".md"):
        return "ERROR: notes are markdown files, such as oat-milk.md."
    candidate = (NOTES / cleaned).resolve()
    if candidate.parent != NOTES.resolve():
        return "ERROR: name escapes the notes folder."
    if not candidate.is_file():
        return "ERROR: " + cleaned + " is not in the catalog."
    return candidate


def _note_title(text):
    for line in text.splitlines():
        if line.startswith("# "):
            return line[2:].strip()
    return "(untitled)"


def list_notes(args):
    """Titles and sizes only. Bodies stay on disk until read_note."""
    del args
    lines = ["NOTES"]
    for path in sorted(NOTES.glob("*.md")):
        text = path.read_text(encoding="utf-8")
        lines.append(
            "- " + path.name + " | " + _note_title(text) + " | " + str(len(text)) + " chars"
        )
    return "\n".join(lines)


def read_note(args):
    found = _note_path(args.get("name", ""))
    if isinstance(found, str):
        return found
    text = found.read_text(encoding="utf-8")
    if len(text) > PER_NOTE_CAP:
        return (
            "ERROR: " + found.name + " is " + str(len(text))
            + " characters. The per-note cap is " + str(PER_NOTE_CAP)
            + ". File a shorter note."
        )
    return "NOTE: " + found.name + "\n\n" + text


def paste_chars():
    total = 0
    for path in NOTES.glob("*.md"):
        total += len(path.read_text(encoding="utf-8"))
    return total


register(
    "list_notes",
    "List huddle notes by name, title, and size. Does not return bodies.",
    {},
    [],
    list_notes,
)
register(
    "read_note",
    "Read one short note by file name, such as oat-milk.md. "
    "A note over the per-note cap returns ERROR. The picnic note is an example.",
    {"name": {"type": "string", "description": "File name from list_notes."}},
    ["name"],
    read_note,
)


## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
reset_db()
catalog = list_notes({})
print("paste_all_chars:", paste_chars())
print("catalog_chars:", len(catalog))
print("TURN_BUDGET:", TURN_BUDGET)
print("PER_NOTE_CAP:", PER_NOTE_CAP)
print(catalog)
print()
print("oversized note:")
print(read_note({"name": "picnic.md"}))
print()
check(paste_chars() > TURN_BUDGET, "pasting every note exceeds the turn budget")
check(len(catalog) < TURN_BUDGET, "the title catalog fits in the budget")

question = (
    "From the huddle notes, how many oat milk cartons were on hand, "
    "and how many cardamom buns did we bake on Thursday? Check the shelf for OM-32. "
    "Do not load the picnic note."
)
result = run_agent(question, system_text())
print("ANSWER:", result["text"])
joined = "\n".join(result["tool_log"])
check(result["stopped"] == "final", "context run finished")
check("oat-milk.md" in joined and "thursday-buns.md" in joined, "the relevant notes were read")
check("picnic.md" not in joined, "the picnic note was not loaded into the loop")
check("3" in result["text"] and "24" in result["text"], "both counts are in the answer")
check("query_inventory" in joined, "the shelf tool from tutorial 2 still runs")
